# SPX calibration smiles with bid–ask spreads

For each of the ten observed maturities, plot the implied-volatility bid–ask range and the fitted IVs from run 005 of the piecewise-constant and Legendre studies. Each figure uses the Reference, Piecewise constant and Legendre legend labels. The saved SPX data contain no non-calibrated maturities.


SPX inputs and numerical results are not distributed. Supply your own data as described in the repository README.


In [ ]:
from pathlib import Path
import pickle
import sys

# Open from anywhere inside this repository.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "wiener_chaos/experiments/reporting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the Wiener chaos pricing repository.")
sys.path.insert(0, str(ROOT))
from wiener_chaos.experiments.reporting import saved_run, save_spx_spread_smile, set_show_figures

DATA_DIR = ROOT / "Real data"
OUTPUT = DATA_DIR / "Plots"
RUN_ID = 5
PIECEWISE_STUDY = DATA_DIR / "Piecewise constant/Statistics calibration/study"
LEGENDRE_STUDY = DATA_DIR / "Legendre/Statistics calibration/study"
DATA_PATH = DATA_DIR / "real_data.pkl"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "SPX inputs and saved results are private. Supply your own quotes, "
        "then run the data and calibration notebooks; see README.md."
    )
with DATA_PATH.open("rb") as handle:
    data = pickle.load(handle)
piecewise = saved_run(PIECEWISE_STUDY, RUN_ID)
legendre = saved_run(LEGENDRE_STUDY, RUN_ID)
set_show_figures(True)
print(f"SPX run {RUN_ID:03d}; output: {OUTPUT}")

In [ ]:
if any(len(data["maturities"]) != len(run["iv_model"])
       for run in (piecewise, legendre)):
    raise ValueError("The saved run and SPX surface have different maturity counts.")
LEGEND_FONT_SIZE = 10
for maturity, strikes, bid_iv, ask_iv, piecewise_iv, legendre_iv in zip(
        data["maturities"], data["strikes"], data["iv_bid"],
        data["iv_ask"], piecewise["iv_model"], legendre["iv_model"]):
    maturity_label = f"{maturity:.5f}".rstrip("0").rstrip(".")
    path = save_spx_spread_smile(
        strikes, bid_iv, ask_iv, [piecewise_iv, legendre_iv],
        ["Piecewise constant", "Legendre"], data["S0"],
        OUTPUT / f"iv_calibration_smile_{maturity_label}.pdf",
        show_legend=True, legend_fontsize=LEGEND_FONT_SIZE,
    )
    print("saved", path.relative_to(ROOT))
print("Run-level IV MAE against observed mid IV (bp):",
      {label: round(run["iv_calibration_metrics"]["mae"] * 1e4, 2)
       for label, run in [("Piecewise constant", piecewise), ("Legendre", legendre)]})